In [ ]:
!pip install -q torch-geometric
import urllib.request
import torch
import time
import psutil
import gc
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader

print("--- NAIVE (BASELINE) PRION PIPELINE TEST ---")
print(f"Initial System RAM: {psutil.virtual_memory().percent}%\n")

pdb_id = "1qlx"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
response = urllib.request.urlopen(url)
pdb_data = response.read().decode('utf-8')

prion_coords = []
for line in pdb_data.split('\n'):
    if line.startswith("ATOM"):
        x = float(line[30:38].strip())
        y = float(line[38:46].strip())
        z = float(line[46:54].strip())
        prion_coords.append([x, y, z])

num_atoms = len(prion_coords)
print(f"Successfully parsed {num_atoms} atoms from the Prion structure.")

num_frames = 10000

class NaivePrionDataset(Dataset):
    def __init__(self, coords, num_frames):
        super().__init__(root=None, transform=None, pre_transform=None)
        self.coords = torch.tensor(coords, dtype=torch.float32)
        self.num_frames = num_frames
        
    def len(self):
        return self.num_frames
        
    def get(self, idx):
        # The naive approach researchers use: yielding an individual PyG Data object per frame
        pos = self.coords + torch.randn_like(self.coords) * 0.1
        return Data(pos=pos, num_nodes=self.coords.shape[0])

dataset = NaivePrionDataset(prion_coords, num_frames)

batch_size = 64
# We use standard num_workers=4, exactly how a researcher would normally write it
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nPushing Data objects to {device.type.upper()}...")

start_sim = time.time()
if torch.cuda.is_available():
    torch.cuda.synchronize()

frames_processed = 0
try:
    for i, batch in enumerate(loader):
        if i % 10 == 0:
            print(f"Batch {i} loaded. RAM: {psutil.virtual_memory().percent}%")
            
        batch_gpu = batch.to(device)
        
        if torch.cuda.is_available():
            # Dummy math simulating the Equivariant NN
            dummy_weights = torch.randn((batch.pos.shape[0], num_atoms, num_atoms), device=device)
            # Not exact bmm due to batching structure, just simulate workload
            torch.cuda.synchronize()
            
        frames_processed += batch.pos.shape[0] // num_atoms
except Exception as e:
    print(f"\nCRASH DETECTED: {e}")

end_sim = time.time()

print("\n" + "="*50)
print("NAIVE PRION SIMULATION RESULTS")
print("="*50)
print(f"Total Frames Processed : {frames_processed}")
print(f"Simulation Time        : {end_sim - start_sim:.2f} seconds")
if frames_processed > 0:
    print(f"Throughput             : {frames_processed / (end_sim - start_sim):.2f} frames/sec")
print(f"System RAM Peak        : {psutil.virtual_memory().percent}%")
print("="*50)
